# HireSignal Matcher: QLoRA fine-tuning of Qwen/Qwen3-4B-Instruct-2507 (Kaggle)

**Before running:** Settings (right panel) -> Accelerator **GPU T4 x2**, Internet **On**, and add the private dataset
`hiresignal-finetune` (Add Input). Run the cells **one at a time, top to bottom**. Each cell says what success looks like.

Training data: human-labelled real Djinni CV/job pairs (+ capped ContractNLI and synthetic bench-v1 shares).
Never trained on: the 202 val requirements, the real eval split, the sealed test (not in the bundle).
Code status: written and tested on CPU with fake labels; not yet run on a Kaggle GPU.

## 0. GPU check
Success: one or two lines `Tesla T4 ... 15.x GiB`. If it says *No GPU*, change the Accelerator and restart.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2, then Run -> Restart session'
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} | free {free/2**30:.1f} / {total/2**30:.1f} GiB | capability {torch.cuda.get_device_capability(i)}')
print('torch', torch.__version__, 'cuda', torch.version.cuda)

## 1. Install packages
Pinned to the versions this code was written for. Success: the last line prints four versions. If pip reports a conflict with Kaggle's torch, re-run with the pins removed and send me the error.

In [ ]:
!pip install -q "transformers==5.19.0" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0" pydantic
import transformers, peft, bitsandbytes, accelerate
print(transformers.__version__, peft.__version__, bitsandbytes.__version__, accelerate.__version__)

## 2. Unpack the bundle and check the data
Success: every file prints `OK`, and the leakage audit shows 0 exact CV duplicates for val, test and eval_real.

In [ ]:
import os, shutil, json, hashlib
src = '/kaggle/input/hiresignal-finetune'
zips = [os.path.join(r, f) for r, _, fs in os.walk(src) for f in fs if f.endswith('.zip')]
if zips:
    shutil.unpack_archive(zips[0], '/kaggle/working/hs')
else:
    shutil.copytree(src, '/kaggle/working/hs', dirs_exist_ok=True)
os.chdir('/kaggle/working/hs')
m = json.load(open('finetuning/data/manifest.json'))
for f, info in m['files'].items():
    h = hashlib.sha256(open('finetuning/data/' + f, 'rb').read()).hexdigest()
    print('OK  ' if h == info['sha256'] else 'SHA MISMATCH', f, {k: v for k, v in info.items() if k not in ('sha256', 'use')})
print(json.dumps({k: m[k] for k in ('labels', 'splits_by_pair', 'extra_train_sources')}, indent=1))
print(json.dumps(m['leakage_audit'], indent=1))
assert not os.path.exists('benchmark/data/test.jsonl'), 'sealed test must not be in the bundle'

## 3. Settings (reproducibility)
`BASE_REV` pins the exact base model commit. Change nothing here unless I ask you to.

In [ ]:
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
BASE_REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
OUT = '/kaggle/working/qwen3-4b-hs-lora'
EVAL = '/kaggle/working/eval'
D = 'finetuning/data'
RUNS = {  # name -> (cases file with gold, prompts file)
    'val_none':  ('benchmark/data/val.jsonl', f'{D}/eval/val_none_prompts.jsonl'),
    'val_rag':   ('benchmark/data/val.jsonl', f'{D}/eval/val_rag_prompts.jsonl'),
    'real_none': (f'{D}/eval/real_cases.jsonl', f'{D}/eval/real_none_prompts.jsonl'),
    'real_rag':  (f'{D}/eval/real_cases.jsonl', f'{D}/eval/real_rag_prompts.jsonl'),
}
import os; os.environ['PYTHONHASHSEED'] = '13'
import subprocess, sys
def evaluate(tag, adapter=None, limit=0):
    for name, (cases, prompts) in RUNS.items():
        if not os.path.exists(prompts):
            print('skip', name, '(not in this bundle: real eval needs labels)'); continue
        cmd = [sys.executable, '-m', 'finetuning.eval_matcher', '--base', BASE, '--revision', BASE_REV, '--cases', cases,
               '--prompts', prompts, '--run-id', f'{tag}_{name}', '--out', EVAL, '--limit', str(limit)]
        subprocess.run(cmd + (['--adapter', adapter] if adapter else []), check=True)

## 4. Smoke test (5 prompts)
Checks that the model downloads and answers in JSON before spending GPU hours. Success: lines like
`{"run": "smoke_val_none", "n": 5, "macro_f1": ..., "json_validity": ...}`. The base download is about 8 GB.

In [ ]:
evaluate('smoke', limit=5)

## 5. Baseline: untuned model (runs A and B)
Same 4-bit load, greedy decoding, 192 new tokens and validator as after training. This can take hours; the cell
is **resumable**: if the session stops, run cells 2-3 and this one again and it continues where it stopped.

In [ ]:
evaluate('base')

## 6. QLoRA training
Success: `trainable params: ~33M`, then a loss line every 5 steps and an `eval_loss` every 20 steps; at the end a
line with `"adapter": ".../adapter"`. Resumable from the last checkpoint.

**If you see `CUDA out of memory`:** Run -> Restart session, run cells 0-3, then run this cell with
`--max-len 1536` added (examples longer than that are skipped and counted). If it still fails, add `--r 8 --alpha 16`.

In [ ]:
!python -m finetuning.train_qlora --base {BASE} --revision {BASE_REV} --data {D} --out {OUT} --epochs 2 --seed 13

## 7. Fine-tuned model (runs C and D) on the same prompts

In [ ]:
evaluate('lora', adapter=f'{OUT}/adapter')

## 8. Comparison
Keep the adapter only if, on **both** val and real, macro-F1 is higher and false-MET is not higher than the base,
with JSON validity not lower. Otherwise we investigate; a worse result is reported as a worse result.

In [ ]:
import json, pandas as pd
rows = []
for tag in ('base', 'lora'):
    for name in RUNS:
        p = f'{EVAL}/{tag}_{name}/metrics.json'
        if not os.path.exists(p):
            continue
        x = json.load(open(p)); f = x['final']; pc = f['per_class']
        rows.append({'model': tag, 'set': name, 'n': x['n'], 'macro_F1': f['macro_f1'], 'false_MET': f['false_met_rate'],
                     **{f'{c}_P': pc[c]['precision'] for c in pc}, **{f'{c}_R': pc[c]['recall'] for c in pc},
                     'JSON_valid': x['json_schema_validity'], 'quote_valid': x['evidence']['raw_quote_validity'],
                     'evidence_attribution': x['evidence']['attribution_vs_gold'],
                     'UNCERTAIN_rate': sum(f['confusion'][g]['UNCERTAIN'] for g in f['confusion']) / x['n'],
                     'review_rate': x['needs_human_review_rate'], 'p50_s': (x['latency_s'] or {}).get('p50')})
table = pd.DataFrame(rows).round(3)
table.to_csv(f'{EVAL}/comparison.csv', index=False)
table

## 9. Save the adapter, model card and results
Success: `results.zip` and `eval_results.zip` appear in the Output panel (right side). Download both.

In [ ]:
import shutil, json
meta = json.load(open(f'{OUT}/train_meta.json'))
card = f'''---
base_model: {BASE}
library_name: peft
license: apache-2.0
tags: [qlora, hiresignal, matcher]
---
# HireSignal Matcher LoRA for {BASE}

- Base revision: `{BASE_REV}`; 4-bit NF4 (double quant), fp16 compute; LoRA r={meta['args']['r']}, alpha={meta['args']['alpha']}, targets {meta['lora_targets']}.
- Task: one job requirement vs one anonymised CV -> JSON {{decision, basis, evidence, explanation}} (Matcher prompt {m['prompt_version']}).
- Training data: {json.dumps(m['files']['sft_train.jsonl']['by_source'])} examples; human labels on public Djinni data (MIT),
  ContractNLI (CC BY 4.0, Koreeda & Manning 2021), synthetic bench-v1. No private CVs. Data is not included in this repo.
- Steps {meta['steps']}, best checkpoint {meta['best_checkpoint']}. Evaluation: see comparison.csv.
- Screening aid only: no automatic hiring decisions; MISSING/not_documented means absence of evidence.
'''
open(f'{OUT}/adapter/README.md', 'w').write(card)
shutil.copy(f'{OUT}/train_meta.json', f'{OUT}/adapter/train_meta.json')
shutil.copy('finetuning/data/manifest.json', f'{OUT}/adapter/data_manifest.json')
shutil.make_archive('/kaggle/working/results', 'zip', OUT, 'adapter')
shutil.make_archive('/kaggle/working/eval_results', 'zip', EVAL)
print(os.listdir(f'{OUT}/adapter'))

## 10. Upload the adapter to a PRIVATE Hugging Face repo (only after we agree the result is worth keeping)
Needs a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets) holding a Hugging Face **write** token. The token is
read from Secrets and never printed. Uploads only the adapter folder (no CV data).

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
REPO = 'YOUR_HF_USERNAME/hiresignal-qwen3-4b-matcher-lora'   # <- change YOUR_HF_USERNAME
api = HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
api.create_repo(REPO, private=True, exist_ok=True, repo_type='model')
api.upload_folder(folder_path=f'{OUT}/adapter', repo_id=REPO, repo_type='model', commit_message='HireSignal Matcher LoRA')
print('uploaded to private repo', REPO, '| private =', api.model_info(REPO).private)